# Single-qubit PiQC (JAX)

Reproduces the MATLAB `one_qubit_adaptive` / `one_qubit_annealing` demos from [aaronuv/piqc](https://github.com/aaronuv/piqc) with the abstract JAX solver.

Paper-scale settings are `n_traj=1000`, `n_steps=100`, `n_iterations=1000`. The cells below use a smaller grid so the notebook is interactive.

In [ ]:
import jax
import matplotlib.pyplot as plt
import numpy as np

from piqcx import AnnealingConfig, PiQC, SmoothingConfig, configure, devices
from piqcx.systems import single_qubit

configure(precision="float32")
print("devices:", devices())

## Open-system adaptive importance sampling

MATLAB: `T=1`, Lindblad `D=0.01` so Ito `Dtilde=D/2`, `R=0.01`, `Q=10`, drives `X,Y`, `psi0=|+>`, target `(|0>+i|1>)/sqrt(2)`.

In [ ]:
problem = single_qubit(
    time=1.0,
    diffusion=0.5 * 0.01,
    control_cost=0.01,
    terminal_weight=10.0,
    terminal_cost="fidelity",
)
print(problem.n_qubits, problem.n_controls, problem.dim)

open_result = PiQC(
    problem,
    n_traj=128,
    n_steps=80,
    n_pulses=40,
    n_iterations=40,
    smoothing=SmoothingConfig(kind="window", window=1, window_after=8, window_late=8),
    seed=0,
).run()
print("final fidelity", float(open_result.fidelity[-1]))

In [ ]:
t = np.linspace(open_result.dt, open_result.time, open_result.n_steps)
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
axes[0, 0].plot(open_result.fidelity)
axes[0, 0].set_ylabel("F")
axes[0, 1].plot(open_result.ess)
axes[0, 1].set_ylabel("ESS")
axes[1, 0].plot(t, open_result.controls.T)
axes[1, 0].set_ylabel("u(t)")
axes[1, 1].plot(open_result.fluence)
axes[1, 1].set_ylabel("fluence")
fig.tight_layout()

## Annealed PiQC (closed-system limit)

Toggle `AnnealingConfig.schedule` among `linear`, `inverse`, `inverse_square`, `exponential`, `steps`, `log_steps`, `const_plus_exp`.

In [ ]:
schedules = ("exponential", "linear", "steps")
annealed = {}
for name in schedules:
    annealed[name] = PiQC(
        problem.with_updates(control_cost=1.0, terminal_weight=100.0),
        n_traj=128,
        n_steps=80,
        n_pulses=40,
        n_iterations=40,
        annealing=AnnealingConfig(enabled=True, schedule=name, d_init=1e-1, d_final=1e-8, n_plateaus=8),
        smoothing=SmoothingConfig(kind="window", window=1, window_after=10, window_late=8),
        seed=1,
    ).run()
    print(name, float(annealed[name].fidelity[-1]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for name, res in annealed.items():
    axes[0].plot(res.fidelity, label=name)
    axes[1].semilogy(res.diffusion_schedule, label=name)
axes[0].set_title("fidelity")
axes[1].set_title("D schedule")
axes[0].legend()
axes[1].legend()
fig.tight_layout()